# Save RNA/ATAC count matrix

In [1]:
suppressPackageStartupMessages({
    library(Seurat)
    library(repr)
    library(patchwork)
    library(ggplot2)
    library(Signac)
    library(tidyverse)
    library(GenomicRanges)
    library(edgeR)
    library(SingleCellExperiment)
    library(Matrix)
    library(scran)
    library(scater)
    library(ggrepel)
    library(fs)
    library(tidyverse)
    library(randomForest)
    library(reticulate)
    library(pheatmap)
    library(gridExtra)
    library(RColorBrewer)
    library(MAST)
    library(data.table)
    library(ComplexHeatmap)
})
options(future.globals.maxSize = Inf)
options(Seurat.object.assay.version = "v5")
options(ggrepel.max.overlaps = Inf)

In [2]:
root_dir <- "/tscc/projects/ps-epigen/users/biy022/MGH/data/tissue/"
setwd(root_dir)

In [3]:
MGH_Object <- readRDS("combined.all.atac/MGH.ALL.rds")

In [4]:
MGH_Object

An object of class Seurat 
654680 features across 235470 samples within 2 assays 
Active assay: RNA (36601 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: ATAC
 4 dimensional reductions calculated: pca, umap.default, harmony, umap.harmony

In [5]:
nbr_downspl_thre <- 15000
nbr_subclass <- table(MGH_Object$Subclass)
subclass_to_downspl <- names(nbr_subclass[nbr_subclass >= nbr_downspl_thre])
removed_cells <- c()
for (subclass in subclass_to_downspl) {
    subclass_barcodes <- colnames(MGH_Object)[MGH_Object$Subclass == subclass]
    tmp_cells <- sample(subclass_barcodes, size = nbr_subclass[subclass] - 15000, replace = FALSE)
    removed_cells <- c(removed_cells, tmp_cells)
}

In [6]:
MGH_Object_downspl <- MGH_Object[, !colnames(MGH_Object) %in% removed_cells]

In [7]:
MGH_Object_downspl

An object of class Seurat 
654680 features across 124380 samples within 2 assays 
Active assay: RNA (36601 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: ATAC
 4 dimensional reductions calculated: pca, umap.default, harmony, umap.harmony

In [8]:
setwd(path_join(c(root_dir, "region_combined_analysis/scenicplus/all/")))

In [9]:
writeMM(MGH_Object_downspl@assays$RNA$counts, "MGH_rna_counts.mtx")

NULL

In [10]:
writeMM(MGH_Object_downspl@assays$ATAC$counts, "MGH_atac_counts.mtx")

NULL

In [11]:
write.table(
    MGH_Object_downspl@meta.data, "MGH_meta_data.tsv", 
    col.names = TRUE, row.names = TRUE, quote = FALSE, sep = "\t")

In [12]:
write.table(
    colnames(MGH_Object_downspl), "MGH_barcodes.tsv", 
    col.names = FALSE, row.names = FALSE, quote = FALSE, sep = "\t")

In [13]:
write.table(
    rownames(MGH_Object_downspl[["RNA"]]), "MGH_gene_names.tsv", 
    col.names = FALSE, row.names = FALSE, quote = FALSE, sep = "\t")

In [14]:
write.table(
    rownames(MGH_Object_downspl[["ATAC"]]), "MGH_region_names.tsv", 
    col.names = FALSE, row.names = FALSE, quote = FALSE, sep = "\t")